# fly-lm Stage 1: 파리 커넥톤(45,669뉴런) 문자단위 LM — 파리 배선 vs 무작위 배선

- 배선: flyvis 내장 커넥톤 그대로 고정 (E/I 부호·시냅스 수 측정값)
- 학습: 셀타입 쌍별 세기 alpha 604개 + bias 65개 (flyvis 방식)
- 컨트롤: 동일 엣지 수·동일 자유도, 무작위 배선 (타입 배정도 셔플)
- forward = 희소행렬 곱 → 그래프 메모리 시점당 ~20MB → TBPTT 96 가능

실행 전: 런타임 → 런타임 유형 변경 → **T4 GPU**

In [ ]:
!pip install -q flyvis 2>&1 | tail -1
import torch
assert torch.cuda.is_available(), 'T4 GPU로 바꾸고 재시작'
device = 'cuda'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 커넥톤 로드 + 셀타입 추출
import numpy as np
from flyvis.connectome import ConnectomeFromAvgFilters

conn = ConnectomeFromAvgFilters(file='fib25-fib19_v2.2.json', extent=15)
nodes, edges = conn.nodes, conn.edges
n = len(nodes.index[:])
src = np.asarray(edges.source_index[:], dtype=np.int64)
tgt = np.asarray(edges.target_index[:], dtype=np.int64)
syn = np.asarray(edges.n_syn[:], dtype=np.float32)
sgn = np.asarray(edges.sign[:], dtype=np.float32)
node_type = np.asarray(nodes.type[:])
role = np.asarray(nodes.role[:])
uniq, type_of_node = np.unique(node_type, return_inverse=True)
n_types = len(uniq)
input_idx = np.nonzero(role == b'input')[0]
output_idx = np.nonzero(role == b'output')[0]
src_type = type_of_node[src]; tgt_type = type_of_node[tgt]
pair_sparse = src_type * n_types + tgt_type
uniq_pairs = np.unique(pair_sparse)
remap = np.full(int(uniq_pairs.max()) + 1, -1, dtype=np.int64)
remap[uniq_pairs] = np.arange(len(uniq_pairs))
pair = remap[pair_sparse]
print(f'n={n:,} E={len(src):,} types={n_types} alpha_pairs={len(uniq_pairs)}')
print(f'input={len(input_idx):,} output={len(output_idx):,}')

In [ ]:
# 모델: 희소행렬 곱 forward (그래프 메모리 O(B×n) per step, not O(B×E))
import torch.nn as nn
import torch.nn.functional as F

class SparseFlyRNN(nn.Module):
    def __init__(self, n, src, tgt, syn, sgn, pair, n_pairs, type_of_node, n_types,
                 input_idx, output_idx, vocab, n_inj=128, n_out=128, seed=42):
        super().__init__()
        self.n = n
        idx = torch.tensor(np.stack([src, tgt]), dtype=torch.int64)
        self.register_buffer('indices', idx)
        self.register_buffer('pair', torch.from_numpy(pair))
        self.log_alpha = nn.Parameter(torch.zeros(n_pairs))
        self.type_bias = nn.Parameter(torch.zeros(n_types))
        self.tau = 0.5
        w0 = torch.log1p(torch.from_numpy(syn))
        self.register_buffer('w0', w0 / w0.max())
        self.register_buffer('sgn', torch.from_numpy(sgn))
        self.register_buffer('node_type', torch.from_numpy(type_of_node.astype(np.int64)))
        g = torch.Generator().manual_seed(seed)
        ii = torch.from_numpy(input_idx.astype(np.int64))
        oo = torch.from_numpy(output_idx.astype(np.int64))
        self.register_buffer('sensory', ii[torch.randperm(len(ii), generator=g)[:n_inj]])
        self.register_buffer('motor', oo[torch.randperm(len(oo), generator=g)[:n_out]])
        self.embed = nn.Parameter(torch.randn(vocab, n_inj) * 0.3)
        self.readout = nn.Parameter(torch.randn(n_out, vocab) * 0.1)

    def forward(self, h, tok):
        inj = self.embed[tok]
        h = h.index_copy(1, self.sensory, torch.tanh(h[:, self.sensory] + inj))
        alpha = torch.clamp(self.log_alpha.exp(), 0, 3)
        vals = self.w0 * self.sgn * alpha[self.pair]           # [E]
        W = torch.sparse_coo_tensor(self.indices, vals, (self.n, self.n)).coalesce()
        acc = torch.sparse.mm(W, h.T).T                        # [B, n]
        bias = self.type_bias[self.node_type].unsqueeze(0)
        h = (1 - self.tau) * h + self.tau * torch.tanh(acc + bias)
        return torch.clamp(h, -10, 10)

    def logits(self, h):
        return h[:, self.motor] @ self.readout

# 무작위 배선 컨트롤 (동일 자유도): 엣지 위치 셔플 + 타입 배정도 셔플
def make_random_wiring(rng, n, src, tgt):
    src2 = rng.permutation(src)
    tgt2 = rng.permutation(tgt)
    m = src2 != tgt2
    return src2[m], tgt2[m]

# 무작위 배선용: 노드 타입도 무작위 재배정 (자유도 동일하게)
def make_random_types(rng, type_of_node, n_types):
    t2 = type_of_node.copy()
    keep = t2[rng.permutation(len(t2))]  # 셔플된 타입 배열
    return keep

In [ ]:
# TinyStories 다운로드 (Colab에서는 전체 받아도 수 분)
import urllib.request, os, time
t0=time.time()
if not os.path.exists('TinyStories-train.txt'):
    urllib.request.urlretrieve(
        'https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-train.txt',
        'TinyStories-train.txt')
print(f'{os.path.getsize("TinyStories-train.txt")/1e6:.0f}MB, {time.time()-t0:.0f}s')
text = open('TinyStories-train.txt', encoding='utf-8', errors='ignore').read(60_000_000)  # 60M chars
CHARS = sorted(set(text)); V = len(CHARS)
c2i = {ch: i for i, ch in enumerate(CHARS)}
tokens = np.array([c2i[ch] for ch in text], dtype=np.int64)
del text
print(f'corpus {len(tokens):,} chars vocab {V}')
from collections import Counter
cnt = Counter(tokens.tolist())
p = np.array([cnt[i] for i in range(V)], dtype=np.float64); p /= p.sum()
UNIGRAM = float(-(p * np.log(p)).sum())
print(f'unigram entropy (bits기준 아님, nats/char): {UNIGRAM:.4f}')

In [ ]:
# 학습 루프 — 파리 배선과 무작위 배선을 같은 함수로
def train(model, tokens, steps=4000, B=16, WARM=64, TBPTT=96, lr=1e-3, tag='model'):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    window = len(tokens) - WARM - TBPTT - 1
    rng = np.random.default_rng(0)
    losses = []
    t0 = time.time()
    for step in range(steps):
        base = int((step * 7 * B) % window)
        idx = (base + np.arange(B) * 9973 + rng.integers(0, 128, size=B)) % window
        batch = torch.tensor(np.stack([tokens[i:i+WARM+TBPTT+1] for i in idx]), device=device)
        h = torch.zeros(B, model.n, device=device)
        with torch.no_grad():
            for t in range(WARM):
                h = model(h, batch[:, t])
        loss = 0.0
        for t in range(WARM, WARM + TBPTT):
            h = model(h, batch[:, t])
            loss = loss + F.cross_entropy(model.logits(h), batch[:, t + 1])
        loss = loss / TBPTT
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        losses.append(loss.item())
        if step % 100 == 0 or step == steps - 1:
            print(f'[{tag}] step {step:5d} loss {loss.item():.4f} [{time.time()-t0:.0f}s]', flush=True)
    return losses

In [ ]:
# 실험 1: 파리 배선
torch.manual_seed(0); np.random.seed(0)
fly = SparseFlyRNN(n, src, tgt, syn, sgn, pair, len(uniq_pairs),
                   type_of_node, n_types, input_idx, output_idx, V)
print('fly params:', sum(p.numel() for p in fly.parameters()))
losses_fly = train(fly, tokens, tag='FLY')
torch.save(fly.state_dict(), 'celltype_fly_t4.pt')

In [ ]:
# 실험 2: 무작위 배선 (동일 엣지 수, 동일 alpha 자유도)
rng = np.random.default_rng(7)
src_r, tgt_r = make_random_wiring(rng, n, src, tgt)
type_r = make_random_types(rng, type_of_node, n_types)
src_type_r = type_r[src_r]; tgt_type_r = type_r[tgt_r]
pair_r_sparse = src_type_r * n_types + tgt_type_r
uniq_r = np.unique(pair_r_sparse)
remap_r = np.full(int(uniq_r.max()) + 1, -1, dtype=np.int64)
remap_r[uniq_r] = np.arange(len(uniq_r))
pair_r = remap_r[pair_r_sparse]
print(f'random wiring: E={len(src_r):,} alpha_pairs={len(uniq_r)}')
rand = SparseFlyRNN(n, src_r, tgt_r, syn[:len(src_r)], sgn[:len(src_r)], pair_r, len(uniq_r),
                    type_r, n_types, input_idx, output_idx, V)
print('rand params:', sum(p.numel() for p in rand.parameters()))
losses_rand = train(rand, tokens, tag='RANDOM')
torch.save(rand.state_dict(), 'celltype_random_t4.pt')

In [ ]:
# 결과 비교
import json
lf, lr_ = np.mean(losses_fly[-50:]), np.mean(losses_rand[-50:])
print(f'\n===== 최종 (last50 평균, 낮을수록 좋음) =====')
print(f'파리 배선   : {lf:.4f}')
print(f'무작위 배선 : {lr_:.4f}')
print(f'unigram     : {UNIGRAM:.4f}')
print(f'파리 - 무작위 = {lf - lr_:+.4f}  ({"파리 우세" if lf < lr_ else "무작위 우세 또는 동급"})')
json.dump({'fly': losses_fly, 'random': losses_rand, 'unigram': UNIGRAM},
          open('t4_results.json', 'w'))

# 학습 곡선
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.plot(np.convolve(losses_fly, np.ones(50)/50, 'valid'), label='fly wiring')
plt.plot(np.convolve(losses_rand, np.ones(50)/50, 'valid'), label='random wiring')
plt.axhline(UNIGRAM, color='gray', ls='--', label='unigram')
plt.xlabel('step'); plt.ylabel('loss (nats/char)'); plt.legend(); plt.title('fly vs random wiring')
plt.savefig('t4_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 보너스: 파리 뇌가 쓴 글 샘플링
i2c = {i: c for c, i in c2i.items()}
@torch.no_grad()
def generate(model, prompt='Once upon a time ', n_chars=300, temperature=0.8):
    model.eval()
    h = torch.zeros(1, model.n, device=device)
    for ch in prompt:
        h = model(h, torch.tensor([c2i.get(ch, 0)], device=device))
    out = list(prompt)
    for _ in range(n_chars):
        lg = model.logits(h) / temperature
        nxt = torch.multinomial(F.softmax(lg, -1), 1)
        out.append(i2c[int(nxt)])
        h = model(h, nxt)
    model.train()
    return ''.join(out)

print('=== 파리 배선 ===')
print(generate(fly))
print('\n=== 무작위 배선 ===')
print(generate(rand))
print('\n=== 결과 다운로드 ===')
from google.colab import files
files.download('t4_results.json'); files.download('t4_curves.png')
files.download('celltype_fly_t4.pt')